# Nail Retouch SDXL LoRA v1

This notebook prepares your processed manicure dataset and trains a first-pass **SDXL LoRA without masks**.

Important: this v1 notebook trains on the **retouched target images plus structured captions** from `metadata.jsonl`. It is the fastest Colab-friendly baseline, and is meant to be used later with **img2img inference** for photo retouching.

## 1. Upload project

Upload the whole repository to Colab, or mount Google Drive and copy the repo to `/content/nail-retouch-assistant`.

Expected files:
- `dataset/processed/train/metadata.jsonl`
- `dataset/processed/val/metadata.jsonl`
- `colab/training_config.yaml`
- `colab/requirements.txt`

In [ ]:
# Optional: mount Drive if you keep the repo there.
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import shutil
import subprocess

PROJECT_ROOT = '/content/nail-retouch-assistant'
CONFIG_PATH = f'{PROJECT_ROOT}/colab/training_config.yaml'
REQUIREMENTS_PATH = f'{PROJECT_ROOT}/colab/requirements.txt'

!python -m pip install -q -r {REQUIREMENTS_PATH}
if not Path('/content/kohya_ss').exists():
    !git clone https://github.com/bmaltais/kohya_ss.git /content/kohya_ss
!python -m pip install -q -r /content/kohya_ss/requirements.txt
!python -m pip install -q xformers

In [ ]:
from pathlib import Path
import json
import yaml

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

project_root = Path(cfg['project_root'])
processed_dir = project_root / cfg['processed_dir']
prepared_dir = Path(cfg['prepared_data_dir'])
output_dir = Path(cfg['output_dir'])
logging_dir = Path(cfg['logging_dir'])

prepared_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
logging_dir.mkdir(parents=True, exist_ok=True)

cfg

In [ ]:
def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

train_rows = load_jsonl(processed_dir / cfg['train_metadata'])
instance_dir = prepared_dir / '100_manicure'
instance_dir.mkdir(parents=True, exist_ok=True)

def export_training_example(row):
    target_src = processed_dir / 'train' / row['target']
    image_name = f"{row['id']}.png"
    image_dst = instance_dir / image_name
    txt_dst = instance_dir / f"{row['id']}.txt"
    shutil.copy2(target_src, image_dst)
    txt_dst.write_text(row['prompt'], encoding='utf-8')

for row in train_rows:
    export_training_example(row)

val_prompt_file = output_dir / 'sample_prompts.txt'
val_prompt_file.write_text('\n'.join(cfg.get('sample_prompts', [])), encoding='utf-8')

print('train examples:', len(train_rows))
print('prepared dir:', instance_dir)
print('sample prompt file:', val_prompt_file)

In [ ]:
import textwrap
import subprocess

train_cfg = cfg['training']
cmd = f"""
cd {cfg['kohya_repo']} && \
accelerate launch --num_cpu_threads_per_process 1 sdxl_train_network.py \
  --pretrained_model_name_or_path={cfg['base_model']} \
  --train_data_dir={prepared_dir} \
  --output_dir={output_dir} \
  --logging_dir={logging_dir} \
  --output_name={cfg['output_name']} \
  --resolution={train_cfg['resolution']} \
  --save_model_as=safetensors \
  --network_module=networks.lora \
  --network_dim={train_cfg['network_dim']} \
  --network_alpha={train_cfg['network_alpha']} \
  --train_batch_size={train_cfg['train_batch_size']} \
  --gradient_accumulation_steps={train_cfg['gradient_accumulation_steps']} \
  --max_train_epochs={train_cfg['max_train_epochs']} \
  --learning_rate={train_cfg['learning_rate']} \
  --unet_lr={train_cfg['unet_lr']} \
  --text_encoder_lr={train_cfg['text_encoder_lr']} \
  --optimizer_type={train_cfg['optimizer_type']} \
  --lr_scheduler={train_cfg['lr_scheduler']} \
  --save_every_n_epochs={train_cfg['save_every_n_epochs']} \
  --mixed_precision={train_cfg['mixed_precision']} \
  --caption_extension=.txt \
  --cache_latents \
  --cache_latents_to_disk \
  --max_data_loader_n_workers={train_cfg['max_data_loader_n_workers']} \
  --bucket_reso_steps={train_cfg['bucket_reso_steps']} \
  --min_bucket_reso={train_cfg['min_bucket_reso']} \
  --max_bucket_reso={train_cfg['max_bucket_reso']} \
  --enable_bucket \
  --xformers \
  --shuffle_caption \
  --persistent_data_loader_workers
"""

cmd = textwrap.dedent(cmd).strip()
print(cmd)

In [ ]:
subprocess.run(cmd, shell=True, check=True, executable='/bin/bash')

## 2. Expected outputs

Training should write files such as:
- `/content/workdir/outputs/nail_retouch_sdxl_v1-000004.safetensors`
- `/content/workdir/outputs/nail_retouch_sdxl_v1-000008.safetensors`

Use those LoRA weights later with **SDXL img2img** at low denoise strength for manicure photo retouching.